# Visualizing Maternal Mortality in New York

In my last post https://medium.com/@billyscardino/addressing-maternal-mortality-in-the-public-health-setting-1970f6109d04

I provided some background on the Maternal Mortality epidemic in the U.S. and New York State.  In the second post of this series I will conduct an exploratory data analysis of Inpatient Maternal Mortality events in New York State by using the Python packages Pandas and Plotly.

## About the datasets:

## Hospital Inpatient Discharges (SPARCS De-Identified): 2015

The Statewide Planning and Research Cooperative System (SPARCS) Inpatient De-identified File contains discharge level detail on patient characteristics, diagnoses, treatments, services, and charges. This data file contains basic record level detail for the discharge. More information about this data can be found here: https://health.data.ny.gov/Health/Hospital-Inpatient-Discharges-SPARCS-De-Identified/82xm-y6g8

## Vital Statistics of New York State

The State Vital Statistics Program includes data from the official records of live births, deaths, fetal deaths, induced terminations of pregnancy/abortions, marriages and divorces/dissolutions of marriage. The Bureau of Vital Records, processes data from live birth, death, fetal death and marriage certificates recorded in New York State outside of New York City.

More information about the Vital Statistics program can be found here: https://www.health.ny.gov/statistics/vital_statistics/vs_program.htm

We will be using the Table 29: Total Pregnancies by Race/Ethnicity and Resident County New York State - 2015 vital statistics tables for our comparison figures for race/ethnicity, which can be found here: https://www.health.ny.gov/statistics/vital_statistics/2015/table29.htm

Let's start by importing the python packages that we'll need in order to conduct our analysis.

In [1]:
from Redshift_connect.AW_connect import AW_engine, AW_connection, lookup_metadata

import pandas as pd
import numpy as np

import dash
import dash_core_components as dcc
import dash_html_components as html
from plotly import tools
from plotly.offline import init_notebook_mode, iplot
import plotly.graph_objs as go
import cufflinks as cf

import colorlover as cl
from IPython.display import HTML

import warnings
warnings.filterwarnings(action='ignore')
init_notebook_mode(connected=True)
cf.set_config_file(world_readable=True,offline=False)

### Methodology to identify Inpatient Maternal Mortality events from SPARCS dataset:

1. Filter the initial dataset to only include pregnant women who had an Inpatient event in New York State in 2015. 

2. Exclude abortions.

3. Keep patient that experienced an 'Expired' disposition (which indicates the patient had a mortality outcome after discharge).

This is the criteria we'll use to count the number of Inpatient Maternal Mortality events.

The SPARCS Hospital inpatient dataset will be accessed by querying an AWS Redshift table and reflecting the data into a Pandas dataframe for analysis.

**Notice the CASE statement for race/ethnicity. This is done to include 'Hispanic' in the category labels.

In [97]:
mm_df = pd.read_sql_query("""
SELECT CASE WHEN race = 'White' and ethnicity = 'Spanish/Hispanic' THEN 'Hispanic'
        WHEN race = 'Other Race' and ethnicity = 'Spanish/Hispanic' THEN 'Hispanic'
        WHEN race = 'Black/African American' THEN 'African American'
        ELSE race
    END race,
    hospital_county,
    health_service_area,
    age_group,
    length_of_stay,
    ccs_diagnosis_description,
    ccs_procedure_description,                 
    apr_medical_surgical_description,
    payment_typology_1,
    emergency_department_indicator,
    total_costs,
    apr_risk_of_mortality
FROM lookup.ip_sparcs 
WHERE gender = 'F'
    AND discharge_year = 2015
    AND apr_mdc_description = 'Pregnancy, Childbirth and the Puerperium'
    AND ccs_procedure_description NOT LIKE 'ABORTION%%'
    AND ccs_diagnosis_description NOT LIKE '%%abortion%%'
    AND patient_disposition = 'Expired' 
;
""",
AW_engine)

In [98]:
# inspect first five rows of the data
display(mm_df.head())

,race,hospital_county,health_service_area,age_group,length_of_stay,ccs_diagnosis_description,ccs_procedure_description,apr_medical_surgical_description,payment_typology_1,emergency_department_indicator,total_costs,apr_risk_of_mortality
0,Hispanic,Queens,New York City,18 to 29,5,Polyhydramnios and other problems of amniotic ...,CESAREAN SECTION,Surgical,Medicaid,Y,9770.72,Minor
1,African American,Nassau,Long Island,30 to 49,2,Polyhydramnios and other problems of amniotic ...,HYSTERECTOMY; AB/VAG,Surgical,Medicaid,N,11638.35,Extreme
2,Other Race,Manhattan,New York City,30 to 49,50,Other complications of pregnancy,INCISION & EXCISION CNS,Surgical,Private Health Insurance,N,447563.86,Extreme
3,African American,Oneida,Central NY,30 to 49,1,Other complications of birth; puerperium affec...,CESAREAN SECTION,Surgical,Medicaid,Y,19487.58,Extreme
4,African American,Manhattan,New York City,30 to 49,3,Other complications of birth; puerperium affec...,INCISION & EXCISION CNS,Surgical,Medicaid,Y,30038.22,Extreme


In [99]:
# describe the dataset attributes
display(mm_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 12 columns):
race                                25 non-null object
hospital_county                     25 non-null object
health_service_area                 25 non-null object
age_group                           25 non-null object
length_of_stay                      25 non-null object
ccs_diagnosis_description           25 non-null object
ccs_procedure_description           25 non-null object
apr_medical_surgical_description    25 non-null object
payment_typology_1                  25 non-null object
emergency_department_indicator      25 non-null object
total_costs                         25 non-null float64
apr_risk_of_mortality               25 non-null object
dtypes: float64(1), object(11)
memory usage: 2.4+ KB


None

There were more than 25 Maternal Mortality events in New York State in 2015, which is the universe to begin the analysis.

Next, we will import the Vital Statistics data using the popular data manipulation package known as Pandas and tidy the data for a comparison analysis.

In [121]:
# import the data
vs_df = pd.read_excel("R:\CPM\Personal Folders\WScardino\Blog\data\Table 29 Total Pregnancies by RaceEthnicity and Resident County New York State 2015.xlsx")

# transpose and reset the index
vs_df = vs_df.T.reset_index()

# rename the columns 
vs_df.columns = ['Race','Total Pregnancies']
vs_df.Race.replace(to_replace={'Black':'African American'}, inplace=True)
display(vs_df)

,Race,Total Pregnancies
0,White,139971
1,African American,68557
2,Other Race,38579
3,Hispanic,80078


Now that we have the data, let's compare the percentage of women in the Inpatient Maternal Mortality cohort to the percentage of women that were pregnant in New York State in 2015 by race/ethnicity.

In [586]:
# create the mortality Count column
mm_df['Count'] = 1

# create the grouped dataframe summing the count column by hospital county
grouped_df = mm_df.groupby(['race']).Count.sum()

# reset the pandas dataframe inndex
grouped_df = grouped_df.reset_index()

merged_df = grouped_df.merge(vs_df,how='left',left_on='race',right_on='Race').drop('race',axis=1)
merged_df.columns = ['Maternal Mortality','Race','Total Pregnancies']
merged_df['Maternal Mortality %'] = round(100 * merged_df['Maternal Mortality'] / merged_df['Maternal Mortality'].sum())
merged_df['Total Pregnancies %'] = round(100 * merged_df['Total Pregnancies'] / merged_df['Total Pregnancies'].sum())
merged_df['Maternal Mortality %'] = merged_df['Maternal Mortality %'].astype(int)
merged_df['Total Pregnancies %'] = merged_df['Total Pregnancies %'].astype(int)

In [588]:
trace1 = go.Bar(
    x=merged_df['Race'],
    y=merged_df['Maternal Mortality %'],
    text=merged_df['Maternal Mortality %'],
    textposition = 'auto',
    name='% of total inpatient<br>maternal mortality events',
    hoverinfo='name+text',
    hoverlabel=dict(namelength=100),
    marker=dict(color='rgb(49,130,189)'),
)

trace2 = go.Bar(
    x=merged_df['Race'],
    y=merged_df['Total Pregnancies %'],
    text=merged_df['Total Pregnancies %'],
    textposition= 'auto',
    name='% of total pregnancies',
    hoverinfo='name+text',
    hoverlabel=dict(namelength=100),
    marker=dict(color='rgb(204,204,204)'),
)

data = [trace1,trace2]

layout = go.Layout(
    legend=dict(font=dict(size=11,
                         family='sans-serif')),
    height=600,
    yaxis=dict(showgrid=False, title='% of Race/Ethnicity'),
    xaxis=dict(title='Race/Ethnicity Categories'),
    title='Comparison between % of Inpatient Maternal Mortality Events and % of Pregnancies by Race in NYS in 2015',
    font=dict(family='sans-serif', size=13)
)

fig = go.Figure(data=data, layout=layout)

iplot(fig, filename='grouped-bar-direct-labels')

40% of the women that experienced Inpatient Maternal Mortality events in New York State in 2015 were African American, which is the highest percentage of the Total Inpatient Maternal Mortality events of all the race/ethnicity categories in the cohort. In addition, African Americans also experienced the highest difference between the percentage of Inpatient Maternal Mortality events and the percentage of Pregnancies in New York State in 2015. While African Americans accounted for 21% of Pregnancies in New York State in 2015, African Americans accounted for 40% of Inpatient Maternal Mortality events in New York State in 2015, which is a difference of 19 percentage points.

'Other Race' was the only other category with a higher percentage of Inpatient Maternal Mortality, compared to the percentage of Pregnancies in New York in 2015.  According to New York State DOH, "Members of multiple races, Native Americans, and Unknown race/ethnicity are assigned to the category ‘Other Race/Ethnicity’." 

More information on these race/ethnicity descriptions can be found here:
https://www.health.ny.gov/health_care/managed_care/reports/docs/demographic_variation/demographic_variation_2015.pdf

Next, let's look at the New York state counties where the Inpatient Maternal Mortality events took place in 2015.

In [581]:
# create the grouped dataframe summing the count column by hospital county
grouped_df = mm_df.groupby('hospital_county')['Count'].sum()
grouped_df = grouped_df.reset_index()
grouped_df2 = mm_df.groupby('hospital_county')['total_costs'].sum()
grouped_df2 = grouped_df2.reset_index()
grouped_df2['total_costs'] = round(grouped_df2['total_costs'])
merged_df = grouped_df.merge(grouped_df2,on='hospital_county')
merged_df.columns = ['Hospital County','Maternal Mortality','Total Costs']
merged_df['Maternal Mortality %'] = round(100 * merged_df['Maternal Mortality'] / merged_df['Maternal Mortality'].sum())

In [584]:
hospital_counties = ['Queens','Manhattan','Albany','Bronx','Kings','Onondaga','Nassau','Oneida','Erie'][::-1]
hospital_counties2 = ['Queens','Manhattan','Albany','Bronx','Kings','Onondaga','Nassau','Oneida','Erie'][::-1]
mm_pct = ['28%','16%','12%', '12%','8%', '8%', '8%', '4%', '4%' ][::-1]
tc = ['$138,038','$1,019,544','$265,916','$91,056','$87,134','$34,248','$44,268','$19,488','$17,342'][::-1]

In [583]:
trace0 = go.Bar(
    x=mm_pct,
    y=hospital_counties,
    text=mm_pct,
    textposition= 'outside',
    hoverinfo='x+name',
    hoverlabel=dict(namelength=100),
    marker=dict(
        color='rgba(50, 171, 96, 0.6)',
        line=dict(
            color='rgba(50, 171, 96, 1.0)',
            width=1),
    ),
    orientation='h',
    name='% of Total Maternal Mortality events (25 Women)',
)
trace1 = go.Scatter(
    x=tc,
    y=hospital_counties2,
    text=tc,
     textposition= 'bottom center',
    textfont=dict(size=9),
    hoverinfo='y+name+text',
    hoverlabel=dict(namelength=100),
    mode='lines+markers+text',
    line=dict(
        color='rgb(128, 0, 128)'),
    name='Total Costs ($1.7M USD)',
)
layout = dict(
    title='% of Total Maternal Mortality Events and Total Costs in $USD by Hospital County',
    hovermode='closest',
    yaxis=dict(
        zeroline=False,
        showgrid=False,
        showline=False,
         showticklabels=True,
        domain=[0, 0.85],
    ),
    yaxis2=dict(
        zeroline=False,
        showgrid=False,
         showline=True,
        showticklabels=False,
        linecolor='rgba(102, 102, 102, 0.8)',
        linewidth=2,
        domain=[0, 0.85],
    ),
    xaxis=dict(
        zeroline=False,
        showline=False,
        showticklabels=True,
        showgrid=False,
        domain=[0, 0.42],
    ),
    xaxis2=dict(
        zeroline=False,
        showline=False,
        showticklabels=True,
        showgrid=False,
        domain=[0.47, 1],
        side='bottom',
        dtick=500000,
    ),
    legend=dict(
        x=0.029,
        y=1.038,
        orientation="h",
        font=dict(
            size=10,
        ),
    ),
    margin=dict(
        l=100,
        r=20,
        t=70,
        b=70,
    ),
    paper_bgcolor='rgb(248, 248, 255)',
    plot_bgcolor='rgb(248, 248, 255)',
)

# Creating two subplots
fig = tools.make_subplots(rows=1, cols=2, specs=[[{}, {}]], shared_xaxes=False,
                          shared_yaxes=False, vertical_spacing=0.001)

fig.append_trace(trace0, 1, 1)
fig.append_trace(trace1, 1, 2)

fig['layout'].update(layout)
iplot(fig, filename='Servicing County')

This is the format of your plot grid:
[ (1,1) x1,y1 ]  [ (1,2) x2,y2 ]



The majority of Inpatient Maternal Mortality events in New York State in 2015 (about 64%) took place in the New York City service area.

Manhattan hospitals comprised 60% of the Total Costs of Inpatient Maternal Mortality events in New York State in 2015, while hospitals in New York City comprised of 78% of the Total Costs of Inpatient Maternal Mortality events for the cohort.

Finally we will examine the distribution of costs by Inpatient procedure and length of stay.

In [437]:
# remove pluses from length_of_stay
mm_df.length_of_stay = [los.replace("+","") if(los=="120 +") else los for los in mm_df.length_of_stay]

# convert to numeric
mm_df.length_of_stay = pd.to_numeric(mm_df.length_of_stay)

# instantiate an empty list to append to
traces = []

# loop over the diagnosis categories creating separate trace objects for each
for proc in mm_df.sort_values('total_costs',ascending=False)['ccs_procedure_description'].unique():
    # create separate plotly trace instances by procedure so that each procedure has a different color
    trace = go.Scatter(
        x=mm_df[(mm_df.ccs_procedure_description==proc)].total_costs,
        y=mm_df[(mm_df.ccs_procedure_description==proc)].length_of_stay,
        mode='markers',
        name=proc,
        text=round(mm_df[mm_df.ccs_procedure_description==proc].total_costs),
        hoverinfo='text+name',
        hoverlabel=dict(namelength=100),
        marker=dict(
            symbol='circle',
            sizemode='area',
            size=mm_df[mm_df.ccs_procedure_description==proc].total_costs/300,
            line = dict(width=1.5,
                       color='rgb(26,26,26)')
        )    
    )
    traces.append(trace)
    
# create layout object
layout = go.Layout(title='Distribution of Total Costs ($USD) and Length of Stay per Procedure', 
    hovermode='closest',
    yaxis= dict(title='Length of Stay (days)',
        showgrid=False,
        zeroline=False,
        showline=False
    ),
#     showlegend=False,
    xaxis=dict(title='Total Cost $USD',
        showgrid=False,
        showline=False,
    )
)

# create a figure
fig  = go.Figure(data=traces, layout=layout)

# call the api
iplot(fig, filename='Cost Categorical Scatter')

C-sections also had the highest cost disparity for Inpatient procedures prior to a Maternal Mortality outcome in New York in 2015, ranging anywhere between $1,900-515,000. One woman who experienced a C-Section had a 79 day length of stay that resulted in a Maternal Mortality outcome.

Incision and excision of the Central nervous system had the second highest disparity, ranging from  $30,000-447,000. One woman who experieced Incision and excision of the Central nervous system had a length of stay for 55 days that resulted in a Maternal Mortality outcome. 

## Conclusion

#### After analyzing the data, we learned 3 key insights about Inpatient Maternal Mortality events in NYS in 2015. 

#### 1. We learned that African American and Hispanic women were the top race/ethnicity category in the percentage of Inpatient Maternal Mortality events in New York State in 2015; African Americans experienced the highest difference between the percentage of Inpatient Maternal Mortality events and the percentage of Pregnancies in New York State in 2015.

#### 2. We also learned that approximately 64% of Maternal Mortality Inpatient events in New York State in 2015 took place in New York City. 

#### 3. Finally, C-sections possessed the greatest cost disparity for procedures in New York in 2015 prior to a mortality outcome, with a broad range of $513,000 in the costs distribution.